In [1]:
pip install pandas emoji openai tiktoken scikit-learn

  Using cached tiktoken-0.8.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.6 kB)
  Using cached jiter-0.8.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (5.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.9/586.9 kB 6.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.3/454.3 kB 18.6 MB/s eta 0:00:00
Using cached tiktoken-0.8.0-cp312-cp312-macosx_11_0_arm64.whl (982 kB)
Using cached jiter-0.8.2-cp312-cp312-macosx_11_0_arm64.whl (310 kB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import re
import string
import json
import os
import emoji
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn.metrics as sklmetrics
#from google.colab import drive
from sklearn.model_selection import train_test_split


In [11]:
pip install kaggle

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for kaggle: filename=kaggle-1.6.17-py3-none-any.whl size=105786 sha256=436bc0bb7375760fc77686691b06760d6518ee7d73c8b566e15cc476be41aa67
  Stored in directory: /Users/fabiomar/Library/Caches/pip/wheels/46/d2/26/84d0a1acdb9c6baccf7d28cf06962ec80529fe1ad938489983
Successfully built kaggle
Note: you may need to restart the kernel to use updated packages.


In [43]:

#import kagglehub

# Download latest version
#path = kagglehub.dataset_download("thedevastator/coding-questions-with-solutions")

#print("Path to dataset files:", path)

train = pd.read_csv("/Users/fabiomar/Downloads/archive/train.csv")
finedf = train.head(20)


In [ ]:
# LLM vars
openai_key = 'Your KEY here'
model_name = 'gpt-4o-mini-2024-07-18' # see https://platform.openai.com/docs/guides/fine-tuning
question_prompt = """
You are a highly accurate coder system that accepts quastions as input and provide correct solutions. The input text describes a quastion, and your job is to provide an answer
"""


# paths
#os.makedirs('/content/data/conversational_data', exist_ok=True)

#df = pd.read_csv("./data/glaucia.csv")

conversational_llmq_jsonl_path = 'content/data/conversational_data/conversational_llmq.jsonl'

In [45]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [46]:
finedf.shape

(20, 7)

In [47]:
finedf.columns

Index(['problem_id', 'question', 'solutions', 'input_output', 'difficulty',
       'url', 'starter_code'],
      dtype='object')

In [48]:
from sklearn.model_selection import train_test_split

train_j, test_j = train_test_split(finedf, test_size=0.2)

## Fine-Tuning  
We fine-tuned ChatGPT-3.5-Turbo using the training data, aiming to achieve superior performance compared to the standard approach of invoking the OpenAI API GPT-4 model.

In [49]:
# Invoking the API
from openai import OpenAI
client = OpenAI(api_key = openai_key)


### Data Transformation  
Prior to beginning the fine-tuning process, our initial step involves transforming our dataframe into a JSON line file format. This formatted file will serve as the prompt input for the fine-tuning process. Each prompt will encapsulate the question and solutions details of every row. Our anticipated outcome from the fine-tuned model will be the corresponding solution for each question.

In [50]:
# Open the file in write mode
#with open('data/conversationaldata/conversational_jabref_task.jsonl', 'w', encoding='utf-8') as f:
with open(conversational_llmq_jsonl_path, 'w', encoding='utf-8') as f:

    # Iterate over the rows in the DataFrame
    for index, row in train_j.iterrows():

        task_text = row['question']
        # Create the user message by formatting the prompt with the title and body
        ###
        user_message = question_prompt + str(task_text)

        assistant_message = row['solutions'] #{col: row[col] for col in columns_to_classify}
        assistant_message = str(assistant_message)
        # Construct the conversation object
        conversation_object = {
            "messages": [
                {"role": "system", "content": "GitHub Task classification"},
                {"role": "user", "content": user_message},
                {"role": "assistant", "content": assistant_message}
            ]
        }

        # Write the conversation object to one line in the file
        f.write(json.dumps(conversation_object, ensure_ascii=False) + '\n')


## Training file  
With our JSON line file generated, it now serves as the foundational conversation input for our fine-tuned model. We're prepared to upload this training file to the OpenAI API to initiate the training process.

In [51]:
## Uplopading a training file
ft_file_jt = client.files.create(
  file=open(conversational_llmq_jsonl_path, "rb"),
  purpose="fine-tune"
)

ft_file_jt

FileObject(id='file-LzCL4TBHaUC6Az4LSWK6rF', bytes=211806, created_at=1734557109, filename='conversational_llmq.jsonl', object='file', purpose='fine-tune', status='processed', status_details=None)

## Model creation  
At last, the stage is set to create the model, designated with the suffix 'repo-prissueclassifier.'

In [52]:
## Creating a fine-tuned model

ft_model_id_file_path = '/content/fine_tuned_model_id.json'

if os.path.exists(ft_model_id_file_path):
    with open(ft_model_id_file_path, 'r') as f:
        ft_model_object = json.load(f)
        ft_model_id: str = ft_model_object["model_id"]
        print(f"ID file upload: {ft_model_id}")
        ft_job_jt = None

else:
    ft_job_jt = client.fine_tuning.jobs.create(
        training_file=ft_file_jt.id,
        model=model_name,
        suffix= "jt-matching"
    )

    ft_model_id = ft_job_jt.id


jt_ft_model = client.fine_tuning.jobs.retrieve(ft_model_id).fine_tuned_model

if jt_ft_model is None:
    print("Model is being finetuned. Please wait...")

else:
    print(f"Model is finetuned and ready to use: {jt_ft_model}")

Model is being finetuned. Please wait...


In [54]:
# Retrieving the state of a fine-tune
jt_ft_model = client.fine_tuning.jobs.retrieve(ft_job_jt.id).fine_tuned_model
print(jt_ft_model)

ft:gpt-4o-mini-2024-07-18:northern-arizona-university-nau:jt-matching:Afvw14D2


In [42]:
# You can track the progress of your fine-tuning job by listing the lastest events. On our models it took about 3 hours to fine-tune each model
client.fine_tuning.jobs.list_events(fine_tuning_job_id=ft_job_jt.id, limit=20)

SyncCursorPage[FineTuningJobEvent](data=[FineTuningJobEvent(id='ftevent-LCMUwz19ghkPQ3EPEaIMsNa8', created_at=1734549339, level='error', message='Training file has 8 example(s), but must have at least 10 examples', object='fine_tuning.job.event', data={'error_code': 'invalid_n_examples', 'error_param': 'training_file'}, type='message'), FineTuningJobEvent(id='ftevent-3ZQ1Fd38EjXCbe9yccztbqjM', created_at=1734549308, level='info', message='Validating training file: file-ViHSH3zijTJLSMRHTniQNN', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-aTarEWBFBFuomqrNpwnnGnS0', created_at=1734549308, level='info', message='Created fine-tuning job: ftjob-IpxwxqO6EhbRKy1R8cuUEPRH', object='fine_tuning.job.event', data={}, type='message')], object='list', has_more=False)

In [40]:
# TODO: put this under the if condition above
# Retrieving the state of a fine-tune
jt_ft_model = client.fine_tuning.jobs.retrieve(ft_job_jt.id).fine_tuned_model
print(jt_ft_model)

if jt_ft_model is not None and ft_job_jt.id is not None:
    with open('/content/fine_tuned_model_id.json', 'w') as f:
        json.dump({'model_id': ft_job_jt.id}, f)


    #drive.mount('/content/drive')
    #!cp fine_tuned_model_id.json /content/drive/MyDrive/

None


## Fine-tuning results  
The successful fine-tuning of all models was completed using the default of 3 epochs. The process spanned approximately 5 hours; however, variations in processing time might occur due to queue dynamics at any given moment.

## Utilizing Fine-tuned model  
Next, another API from OpenAI is used to invoke the fine-tuned model and assess its performance on the testing dataset.

In [67]:
import openai
import time
import pandas as pd
import re
import concurrent.futures
import tiktoken
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report

# Replace 'open-ai-key' with your actual OpenAI API key
openai.api_key = openai_key
# max_token here should be one since domains are one token long. This might change for future versions of the model and api but you can check the value on the
def query_chatgpt(prompt, model, temperature=0.0,  max_tokens=1, max_retries=5):
    """
    Function to query ChatGPT-4 with a given prompt, with retries for timeouts.

    :param prompt: Prompt string to send to ChatGPT-2.5
    :param model: The model to use, default is ChatGPT-3.5
    :param max_tokens: Maximum number of tokens to generate
    :param max_retries: Maximum number of retries for timeout
    :return: Response from ChatGPT-3.5 or None if all retries fail
    """
    #print('prompt:', prompt)
    attempt = 0
    max_content_tokens = 3999
    encoding = tiktoken.get_encoding("cl100k_base")
    encoding = tiktoken.encoding_for_model(model_name)

    # Function to truncate the message and avoid passing the limit of 4k tokens per gpt-3.5 fine-tuned model limitations
    def truncate_message(message, max_length):
        tokens = encoding.encode(message)
        if len(tokens) > max_length:
            truncated_tokens = tokens[:max_length]
            message = encoding.decode(truncated_tokens)
        return message

    # Truncate the prompt if necessary
    prompt = truncate_message(prompt, max_content_tokens)
    prompt_sample = truncate_message(prompt, 50)
    print('prompt sample:', prompt_sample)

    while attempt < max_retries:
        with concurrent.futures.ThreadPoolExecutor() as executor:
            future = executor.submit(
                openai.chat.completions.create,
                model=model,
                messages=[{"role": "system", "content": question_prompt}, {"role": "user", "content": prompt}],
                max_tokens=max_content_tokens,
                temperature=temperature
            )
            try:
                response = future.result()
                return response.choices[0].message.content
            except concurrent.futures.TimeoutError:
                print(f"Attempt {attempt + 1}/{max_retries} - Request timed out. Retrying...")
            except Exception as e:
                print(f"Attempt {attempt + 1}/{max_retries} - An error occurred: {e}")
            finally:
                attempt += 1

    print("Failed to get a response after several retries.")
    return None

#labels = ['feature', 'bug', 'question']

In [68]:
test_j.head()

,problem_id,question,solutions,input_output,difficulty,url,starter_code
7,7,The only difference between easy and hard vers...,"[""import sys\ndef I():\n return sys.stdin.r...","{\n ""inputs"": [\n ""3\n3\n1 5\n2 10\n2 8\n7...",interview,https://codeforces.com/problemset/problem/1251/E2,NaN
3,3,"You have $n$ barrels lined up in a row, number...","[""def solve():\n n, k = map(int,input().spl...","{\n ""inputs"": [\n ""2\n4 1\n5 5 5 5\n3 2\n0...",interview,https://codeforces.com/problemset/problem/1430/B,NaN
9,9,Alice and Bob play a game. They have a binary ...,"[""for _ in range(int(input())):\n s = input...","{\n ""inputs"": [\n ""5\n01111001\n0000\n1111...",interview,https://codeforces.com/problemset/problem/1398/B,NaN
2,2,"You are given three sequences: $a_1, a_2, \ldo...","[""import sys\nimport random\nfrom fractions im...","{\n ""inputs"": [\n ""5\n3\n1 1 1\n2 2 2\n3 3...",interview,https://codeforces.com/problemset/problem/1408/A,NaN


In [82]:

dataset = []

iterations = len(test_j)

print(f"Number of iterations: {iterations}")
print(f"Model: {jt_ft_model}")
print("---------------------------\n")


# Now let's loop through the test data and classify the GitHub issues
for i in range(iterations):
    
    correct_arr = test_j.iloc[i]['solutions']
    pr_num      = test_j.iloc[i]['problem_id']
    description = test_j.iloc[i]['question']
    #print("pr_num:", pr_num)
    
    prompt = f"Please code a solution for this question: {description}"
    
    response = query_chatgpt(prompt, jt_ft_model, temperature=1.0)

    if response is None:
        print("Failed to get a response after several retries. Skipping this item.")
        continue  # Skip this iteration and move to the next one

    #response_json_str = response.replace("'", '"')
    print('response:', response_json_str)
    #response_json = json.loads(response_json_str)
    #response_arr = list(response_json.values())
    response_arr = response
    # output: list of objects, where each object has two lists
    cur_row = {
        "description": description,
        "correct": correct_arr,
        "predicted": response_arr,
        "pr": pr_num
    }
    dataset.append(cur_row)

    print('--------- result ----------')
    print(pr_num)
    print(description)
    print(correct_arr)
    print(response_arr)
    print('---------------------------')

    time.sleep(6)  # Wait for 6 seconds before retrying


Number of iterations: 4
Model: ft:gpt-4o-mini-2024-07-18:northern-arizona-university-nau:jt-matching:Afvw14D2
---------------------------

prompt sample: Please code a solution for this question: The only difference between easy and hard versions is constraints.

Now elections are held in Berland and you want to win them. More precisely, you want everyone to vote for you.

There are $n$ voters
response: ["from heapq import *\nimport sys\ninput=sys.stdin.readline\ndef find(p,r,va,kt):\n    if(r==0):\n        return va,kt\n    else:\n        tmp=-p[0]\n        val,ind=hp.pop(p,0)\n        if(val>r):\n            hp.push(p,val-r,ind)\n            return find(p,val-r,va+[ind],kt+len(va))\n        else:\n            return find(p,val,va+[ind],kt+len(va)+1)\nline=lambda: list(map(int,input().split()))\ninf=10**18\nfor tt in range(int(input())):\n    n=int(input())\n    mpl=line()\n    pr=line()\n    mpl=[0]+mpl\n    spr=[inf]+pr\n    for i in range(1,n+1):\n        if(mpl[i]<i-1):\n         

In [83]:
dataset

[{'description': "The only difference between easy and hard versions is constraints.\n\nNow elections are held in Berland and you want to win them. More precisely, you want everyone to vote for you.\n\nThere are $n$ voters, and two ways to convince each of them to vote for you. The first way to convince the $i$-th voter is to pay him $p_i$ coins. The second way is to make $m_i$ other voters vote for you, and the $i$-th voter will vote for free.\n\nMoreover, the process of such voting takes place in several steps. For example, if there are five voters with $m_1 = 1$, $m_2 = 2$, $m_3 = 2$, $m_4 = 4$, $m_5 = 5$, then you can buy the vote of the fifth voter, and eventually everyone will vote for you. Set of people voting for you will change as follows: ${5} \\rightarrow {1, 5} \\rightarrow {1, 2, 3, 5} \\rightarrow {1, 2, 3, 4, 5}$.\n\nCalculate the minimum number of coins you have to spend so that everyone votes for you.\n\n\n-----Input-----\n\nThe first line contains one integer $t$ ($1 

In [84]:
df = pd.DataFrame(dataset)

# Specify the output file name
output_file = "./content/data/solutions_predictions-temp10.csv"

# Save the DataFrame to a CSV file
df.to_csv(output_file, index=False)

print(f"CSV file '{output_file}' has been created with the following content:")
print(df)

CSV file './content/data/solutions_predictions-temp10.csv' has been created with the following content:
                                         description  \
0  The only difference between easy and hard vers...   
1  You have $n$ barrels lined up in a row, number...   
2  Alice and Bob play a game. They have a binary ...   
3  You are given three sequences: $a_1, a_2, \ldo...   

                                             correct  \
0  ["import sys\ndef I():\n    return sys.stdin.r...   
1  ["def solve():\n    n, k = map(int,input().spl...   
2  ["for _ in range(int(input())):\n    s = input...   
3  ["import sys\nimport random\nfrom fractions im...   

                                           predicted  pr  
0  ["from heapq import *\nimport sys\ninput=sys.s...   7  
1  ["for _ in range(int(input())):\n    n, k = li...   3  
2  ["for _ in range(int(input())):\n    s = input...   9  
3  ["for _ in range(int(input())):\n    n = int(i...   2  


### Don't run - this is for testing purposes

In [ ]:
### Don't run - this is for testing purposes

import pandas as pd

# Example list containing dictionaries with 'correct', 'predicted', and 'pr' keys
data = [
    {
        "correct": "some correct text",
        "predicted": "some predicted text",
        "pr": 7
    },
    {
        "correct": "another correct text",
        "predicted": "another predicted text",
        "pr": 3
    }
]

dataset_test = [{'correct': '["import sys\\ndef I():\\n    return sys.stdin.readline().rstrip()\\n\\nclass Heap:\\n    def __init__( self ):\\n        self.l = [ -1 ]\\n        self.n = 0\\n    def n( self ):\\n        return self.n\\n    def top( self ):\\n        return self.l[ 1 ]\\n    def ins( self, x ):\\n        self.l.append( x )\\n        n = len( self.l ) - 1\\n        i = n\\n        while i > 1:\\n            j = i // 2\\n            if self.l[ j ] > self.l[ i ]:\\n                self.l[ j ], self.l[ i ] = self.l[ i ], self.l[ j ]\\n                i = j\\n            else:\\n                break\\n    def pop( self ):\\n        r = self.l[ 1 ]\\n        l = self.l.pop()\\n        n = len( self.l ) - 1\\n        if n:\\n            self.l[ 1 ] = l\\n            i = 1\\n            while True:\\n                j = i * 2\\n                k = j + 1\\n                if k < len( self.l ) and self.l[ i ] > max( self.l[ j ], self.l[ k ] ):\\n                    if self.l[ j ] == min( self.l[ j ], self.l[ k ] ):\\n                        self.l[ i ], self.l[ j ] = self.l[ j ], self.l[ i ]\\n                        i = j\\n                    else:\\n                        self.l[ i ], self.l[ k ] = self.l[ k ], self.l[ i ]\\n                        i = k\\n                elif k < len( self.l ) and self.l[ i ] > self.l[ k ]:\\n                    self.l[ i ], self.l[ k ] = self.l[ k ], self.l[ i ]\\n                    i = k\\n                elif j < len( self.l ) and self.l[ i ] > self.l[ j ]:\\n                    self.l[ i ], self.l[ j ] = self.l[ j ], self.l[ i ]\\n                    i = j\\n                else:\\n                    break\\n        return r\\n\\nt = int( I() )\\nfor _ in range( t ):\\n    n = int( I() )\\n    voter = [ list( map( int, I().split() ) ) for _ in range( n ) ]\\n    h = Heap()\\n    d = {}\\n    for m, p in voter:\\n        if m not in d:\\n            d[ m ] = []\\n        d[ m ].append( p )\\n    need = {}\\n    c = 0\\n    sk = sorted( d.keys() )\\n    for m in sk:\\n        need[ m ] = max( 0, m - c )\\n        c += len( d[ m ] )\\n    c = 0\\n    ans = 0\\n    for m in sk[::-1]:\\n        for p in d[ m ]:\\n            h.ins( p )\\n        while c < need[ m ]:\\n            c += 1\\n            ans += h.pop()\\n    print( ans )\\n", "import heapq\\nimport sys\\ninput = sys.stdin.readline\\n\\n\\nt = int(input())\\nfor _ in range(t):\\n    n = int(input())\\n    info = [list(map(int, input().split())) for i in range(n)]\\n    info = sorted(info)\\n    cnt = [0] * n\\n    for i in range(n):\\n        ind = info[i][0]\\n        cnt[ind] += 1\\n    ruiseki_cnt = [0] * (n+1)\\n    for i in range(n):\\n        ruiseki_cnt[i+1] = ruiseki_cnt[i] + cnt[i]\\n    # print(cnt)\\n    # print(ruiseki_cnt)\\n    need = [0] * n\\n    for i in range(1,n):\\n        if cnt[i] != 0 and i > ruiseki_cnt[i]:\\n            need[i] = min(i - ruiseki_cnt[i], i)\\n    # print(need)\\n    info = sorted(info, reverse = True)\\n    #print(info)\\n\\n    num = n - 1\\n    pos = 0\\n    q = []\\n    used_cnt = 0\\n    ans = 0\\n    while True:\\n        if num == -1:\\n            break\\n        while True:\\n            if pos < n and info[pos][0] >= num:\\n                heapq.heappush(q, info[pos][1])\\n                pos += 1\\n            else:\\n                break\\n        if need[num] - used_cnt > 0:\\n            tmp = need[num] - used_cnt\\n            for _ in range(tmp):\\n                ans += heapq.heappop(q)\\n            used_cnt += tmp\\n        num -= 1\\n    print(ans)", "import sys\\ninput = sys.stdin.readline\\n\\nimport heapq\\nfrom itertools import accumulate\\n\\nt=int(input())\\n\\nfor test in range(t):\\n    n=int(input())\\n    M=[[] for i in range(n)]\\n    MCOUNT=[0]*(n)\\n\\n    for i in range(n):\\n        m,p=list(map(int,input().split()))\\n        M[m].append(p)\\n        MCOUNT[m]+=1\\n\\n    #print(M)\\n    #print(MCOUNT)\\n\\n    ACC=list(accumulate(MCOUNT))\\n\\n    #print(ACC)\\n    HQ=[]\\n    ANS=0\\n    use=0\\n\\n    for i in range(n-1,-1,-1):\\n        for j in M[i]:\\n            heapq.heappush(HQ,j)\\n\\n        #print(HQ)\\n            \\n        while ACC[i-1]+use<i:\\n            x=heapq.heappop(HQ)\\n            ANS+=x\\n            use+=1\\n\\n\\n\\n    print(ANS)\\n            \\n            \\n        \\n        \\n        \\n\\n    \\n\\n    \\n", "import sys\\nfrom heapq import heappop, heappush\\n\\nreader = (line.rstrip() for line in sys.stdin)\\ninput = reader.__next__\\n \\nt = int(input())\\nfor _ in range(t):\\n    n = int(input())\\n    mp = []\\n    for i in range(n):\\n        mi, pi = list(map(int, input().split()))\\n        mp.append((mi, pi))\\n    mp.sort()\\n    \\n    prices = []\\n    cost = 0\\n    bribed = 0\\n    i = n - 1\\n    while i >= 0:\\n        currM = mp[i][0]\\n        heappush(prices, mp[i][1])\\n        while i >= 1 and mp[i-1][0] == currM:\\n            i -= 1\\n            heappush(prices, mp[i][1])\\n        already = i + bribed\\n        for k in range(max(0, currM - already)):\\n            cost += heappop(prices)\\n            bribed += 1\\n        i -= 1\\n    \\n    print(cost)\\n", "import sys\\ninput = sys.stdin.readline\\nimport heapq as hq\\nt = int(input())\\nfor _ in range(t):\\n  n = int(input())\\n  vt = [list(map(int,input().split())) for i in range(n)]\\n  vt.sort(reverse=True)\\n  q = []\\n  hq.heapify(q)\\n  ans = 0\\n  cnt = 0\\n  for i in range(n):\\n    hq.heappush(q,vt[i][1])\\n    if vt[i][0] >= n-i+cnt:\\n      ans += hq.heappop(q)\\n      cnt += 1\\n  print(ans)", "import sys\\nimport heapq as hq\\n\\nreadline = sys.stdin.readline\\nread = sys.stdin.read\\nns = lambda: readline().rstrip()\\nni = lambda: int(readline().rstrip())\\nnm = lambda: map(int, readline().split())\\nnl = lambda: list(map(int, readline().split()))\\nprn = lambda x: print(*x, sep=\'\\\\n\')\\n\\ndef solve():\\n    n = ni()\\n    vot = [tuple(nm()) for _ in range(n)]\\n    vot.sort(key = lambda x: (-x[0], x[1]))\\n    q = list()\\n    c = 0\\n    cost = 0\\n    for i in range(n):\\n        hq.heappush(q, vot[i][1])\\n        while n - i - 1 + c < vot[i][0]:\\n            cost += hq.heappop(q)\\n            c += 1\\n    print(cost)\\n    return\\n\\n\\n# solve()\\n\\nT = ni()\\nfor _ in range(T):\\n    solve()\\n", "import sys\\nimport heapq as hp\\n#sys.stdin = open(\'in\', \'r\')\\nt = int(sys.stdin.readline())\\nfor ti in range(t):\\n    n = int(sys.stdin.readline())\\n    a = [tuple(map(int, sys.stdin.readline().split())) for i in range(n)]\\n    a.sort(key = lambda x: (x[0], -x[1]))\\n    c = 0\\n    h = []\\n    res = 0\\n    for i in range(n-1,-1,-1):\\n        hp.heappush(h, a[i][1])\\n        while c + i < a[i][0]:\\n            res += hp.heappop(h)\\n            c += 1\\n    print(res)\\n\\n\\n#sys.stdout.write(\'YES\\\\n\')\\n#sys.stdout.write(f\'{res}\\\\n\')\\n#sys.stdout.write(f\'{y1} {x1} {y2} {x2}\\\\n\')\\n", "import sys\\nfrom heapq import *\\n#sys.stdin = open(\'in\', \'r\')\\nt = int(sys.stdin.readline())\\nfor ti in range(t):\\n    n = int(sys.stdin.readline())\\n    a = [tuple(map(int, sys.stdin.readline().split())) for i in range(n)]\\n    a.sort(key = lambda x: (x[0], -x[1]))\\n    c = 0\\n    h = []\\n    res = 0\\n    for i in range(n-1,-1,-1):\\n        heappush(h, a[i][1])\\n        while c + i < a[i][0]:\\n            res += heappop(h)\\n            c += 1\\n    print(res)\\n\\n\\n#sys.stdout.write(\'YES\\\\n\')\\n#sys.stdout.write(f\'{res}\\\\n\')\\n#sys.stdout.write(f\'{y1} {x1} {y2} {x2}\\\\n\')\\n"]',
  'predicted': '["for _ in range(int(input())):\\n    n = int(input())\\n    li = []\\n    for i in range(n):\\n        x, y = map(int, input().split())\\n        li.append([x, y])\\n    li.sort()\\n    mini = 1e18\\n    maxx = 0\\n    sum1 = 0\\n    for i in range(n):\\n        if maxx < li[i][0]:\\n            mini = min(mini, sum1 + li[i][1])\\n        maxx = max(maxx, li[i][0])\\n        sum1 += li[i][1]\\n    print(min(mini, sum(li[i][1] for i in range(n))))\\n"])',
  'pr': 7},
 {'correct': '["def solve():\\n    n, k = map(int,input().split())\\n    lst = list(map(int,input().split()))\\n    lst.sort()\\n    ans = 0\\n    for i in range(n - k - 1, n):\\n        ans += lst[i]\\n    print(ans)\\nfor i in range(int(input())):\\n    solve()", "t=int(input())\\nfor i in range(t):\\n    n,k=[int(i) for i in input().split()]\\n    a=[int(i) for i in input().split()]\\n    a.sort(reverse=True)\\n    print(sum(a[:k+1]))", "# map(int, input().split())\\nrw = int(input())\\nfor wewq in range(rw):\\n    n, k = list(map(int, input().split()))\\n    a = list(map(int, input().split()))\\n    a.sort()\\n    a.reverse()\\n    f = 0\\n    for i in range(k + 1):\\n        f += a[i]\\n    print(f)\\n", "t=int(input())\\nfor you in range(t):\\n    l=input().split()\\n    n=int(l[0])\\n    k=int(l[1])\\n    l=input().split()\\n    li=[int(i) for i in l]\\n    if(k==0):\\n        print(max(li)-min(li))\\n        continue\\n    z=0\\n    li.sort()\\n    li.reverse()\\n    for i in range(k+1):\\n        z+=li[i]\\n    print(z)\\n", "for _ in range (int(input())):\\n    n,k=map(int,input().split())\\n    a=list(map(int,input().split()))\\n    a.sort(reverse=True)\\n    for i in range (1,k+1):\\n        a[0]+=a[i]\\n        a[i]=0\\n    print(a[0]-a[1])", "for __ in range(int(input())):\\n    n, k = list(map(int, input().split()))\\n    ar = list(map(int, input().split()))\\n    ar.sort(reverse=True)\\n    ans = 0\\n    for i in range(min(n, k + 1)):\\n        ans += ar[i]\\n    print(ans)", "import sys, math\\nimport io, os\\n#data = io.BytesIO(os.read(0,os.fstat(0).st_size)).readline\\nfrom bisect import bisect_left as bl, bisect_right as br, insort\\nfrom heapq import heapify, heappush, heappop\\nfrom collections import defaultdict as dd, deque, Counter\\n#from itertools import permutations,combinations\\ndef data(): return sys.stdin.readline().strip()\\ndef mdata(): return list(map(int, data().split()))\\ndef outl(var) : sys.stdout.write(\'\\\\n\'.join(map(str, var))+\'\\\\n\')\\ndef out(var) : sys.stdout.write(str(var)+\'\\\\n\')\\n#from decimal import Decimal\\n#from fractions import Fraction\\n#sys.setrecursionlimit(100000)\\nINF = float(\'inf\')\\nmod=10**9+7\\n\\n\\nfor t in range(int(data())):\\n    n,k=mdata()\\n    a=sorted(mdata(),reverse=True)\\n    s=sum(a[:k+1])\\n    out(s)\\n", "import sys\\ninput = sys.stdin.readline\\n\\nt = int(input())\\nfor i in range(t):\\n    n,k = map(int,input().split())\\n    a = list(map(int,input().split()))\\n    a.sort()\\n    a.reverse()\\n    cum = [a[0]]\\n    for i in range(n-1):\\n        cum.append(cum[i]+a[i+1])\\n    cum.append(cum[-1])\\n    print(cum[k])", "t = int(input())\\nfor _ in range(t):\\n    #n = int(input())\\n    n, k=map(int, input().split())\\n    a = list(map(int, input().split()))\\n    a.sort()\\n    s=0\\n    for i in range(k+1):\\n        s+=a[n-1-i]\\n    print(s)", "def main():\\n    N, K = list(map(int, input().split()))\\n    *A, = list(map(int, input().split()))\\n    \\n    A.sort()\\n    print(A[-1] + sum(A[-K-1:-1]))\\n\\ndef __starting_point():\\n    for __ in [0]*int(input()):\\n        main()\\n\\n__starting_point()", "import sys\\nimport random\\n# import numpy as np\\nimport math\\nimport copy\\nfrom heapq import heappush, heappop, heapify\\nfrom functools import cmp_to_key\\nfrom bisect import bisect_left, bisect_right\\nfrom collections import defaultdict, deque, Counter\\n# sys.setrecursionlimit(1000000)\\n# input aliases\\ninput = sys.stdin.readline\\ngetS = lambda: input().strip()\\ngetN = lambda: int(input())\\ngetList = lambda: list(map(int, input().split()))\\ngetZList = lambda: [int(x) - 1 for x in input().split()]\\n\\nINF = float(\\"inf\\")\\n\\nMOD = 10 ** 9 + 7\\ndivide = lambda x: pow(x, MOD-2, MOD)\\n\\ndef judge(at, ax, ay, bt, bx, by):\\n    if abs(at - bt) >= abs(ax - bx) + abs(ay - by):\\n        return True\\n    else:\\n        return False\\n\\n\\ndef solve():\\n    n, k = getList()\\n    li = getList()\\n\\n    if k >= n:\\n        print(sum(li))\\n        return\\n\\n    li.sort(reverse=True)\\n    print(sum(li[:k+1]))\\n\\n    return\\n\\ndef main():\\n    n = getN()\\n    for _ in range(n):\\n        solve()\\n\\n    return\\ndef __starting_point():\\n    main()\\n    # solve()\\n\\n__starting_point()", "from sys import stdin\\nt = int(stdin.readline())\\nfor _ in range(t):\\n    n, k = tuple(int(x) for x in stdin.readline().split())\\n    lst = sorted(int(x) for x in stdin.readline().split())\\n    print(sum(lst[-k-1:]))\\n", "t = int(input())\\nfor _ in range(t):\\n    n,k = [int(x) for x in input().split()]\\n    l = [int(x) for x in input().split()]\\n    l.sort()\\n    l.reverse()\\n    print(sum(l[:min(k+1,n)]))", "for _ in range(int(input())):\\n\\tn, k = list(map(int, input().split()))\\n\\tA = list(map(int, input().split()))\\n\\n\\tA.sort(reverse=True)\\n\\tif k == 0:\\n\\t\\tprint(max(A) - min(A))\\n\\telse:\\n\\t\\tprint(A[0] + sum(A[1:k+1]))\\n", "n = int(input())\\n\\nfor _ in range(n):\\n    n, k = list(map(int, input().split()))\\n    arr = list(map(int, input().split()))\\n    arr.sort(reverse=True)\\n\\n    print(sum(arr[:k+1]))\\n", "\\"\\"\\"T=int(input())\\nfor _ in range(0,T):\\n    n=int(input())\\n    a,b=map(int,input().split())\\n    s=input()\\n    s=[int(x) for x in input().split()]\\n    for i in range(0,len(s)):\\n        a,b=map(int,input().split())\\"\\"\\"\\n\\n\\nT=int(input())\\nfor _ in range(0,T):\\n    n,k=list(map(int,input().split()))\\n    s=[int(x) for x in input().split()]\\n    s.sort()\\n    s=s[::-1]\\n    for i in range(1,min(k+1,len(s))):\\n        s[0]+=s[i]\\n\\n    print(s[0])\\n", "t=int(input())\\nwhile t:\\n\\tt-=1\\n\\tn,k=list(map(int,input().split()))\\n\\ta=[int(i) for i in input().split()]\\n\\ta.sort()\\n\\tans=0\\n\\ta.reverse()\\n\\tfor i in range(k+1):\\n\\t\\tans+=a[i]\\n\\t\\t\\n\\tprint(ans)\\n"]',
  'predicted': '["from sys import stdin,stderr\\ndef msg(*args,**kwargs):\\n    print(*args,**kwargs,file=stderr)\\n#I love the STL\\nimport math,itertools,functools,string,heapq,bisect,collections,time\\n#I love the Excepton Handling\\ntry:\\n    #I love the Exception Handling\\n    #I love the self-testing\\n    from prg import *\\n    __test=false\\nexcept ImportError:\\n    __test=true\\nclass Mnemonics:\\n    def __init__(self,base=10):\\n        self.base=base\\n        self.Char=collections.namedtuple(\'Char\',\'value,inv,code\')\\n        self.memo={}\\n        self.CharCode={}\\n        self.CharCodeInv={}\\n        self.index={}\\n        self.index_inv={}\\n        self.size=0\\n\\n    def register(self,char,value):\\n        \'\'\'register char for the base\'\'\'\\n        if char in self.CharCode:\\n            return\\n        inv=value self.base\\n        code=\'\'%s\'\'%%{0:self.base}\'\'%%(inv)\'\'\'%self.Char  \\n        self.CharCode[code]=self.Char(value,inv,char)\\n        self.CharCodeInv[char]=self.Char(value,inv,code)\\n        self.index[self.size]=char\\n        self.index_inv[char]=self.size\\n        self.size+=1\\n\\n    def init_from_list(self,CharList):\\n        \'\'\'init from list of tuple (char,value)\'\'\'\\n        for char,value in CharList:\\n            self.register(char,value)\\n\\n    def init_from_string(self,string,value_from_index):\\n        \'\'\'it will treat string as the base:index\'\'\'\\n        self.init_from_list([(string[i],value_from_index(i)) for i in range(len(string))])\\n\\n    def get_mnemonic(self,index):\\n        \'\'lexicographically get from index\'\'\\n        if index_in(self,index):\\n            return \'\'%%(index)\'\'%%%self.get_base()\'\'%%{\'\'0\'\'}}\'\'%%(index)\'\'\'%self.Char  \\n        s=\'\'\'\'\\n        while index >= self.size:\\n            index  self.size\'\'\\n            s+=self.get_mnemonic(index)\\n        return s+self.index[index]\\n\\n    def get_inverse(self,char):\\n        return self.CharCodeInv[char].inv\\n    def get_value(self,char):\\n        return self.CharCodeInv[char].value\\n    def get_inverse_from_index(self,index):\\n        return self.index_inv[self.get_mnemonic(index)]\\n    def get_value_from_index(self,index):\\n        return self.index[self.get_mnemonic(index)]\\n    def get_mnemonic_from_index(self,index):\\n        return self.get_mnemonic(index)\\n\\n    def get_base(self):\\n        return self.base\\n    def base_increase(self):\\n        self.base+=1\\n        for i in range(self.size):\\n            self.index[i]=self.get_mnemonic(i)\\n   \\nbase=2\\nChar=[\'0\',\'1\']\\ninc_table=[lambda x:base(x)(x+1),lambda x:base(x)(0),lambda x:base(x)(x+1)]\\ndef pourings(n,k,a):\\n    x=base(k)(a)\\n    f_max=lambda x:inc_table[x][(x+1)%base]\\n    f_min=lambda x:inc_table[x][0]\\n    s=list(map(lambda x:f_max(x),x))\\n    s[:k]=sorted(s[:k])\\n    return s\\npourings(3,3,[1,2,3])\\nexit()\\n\\nmnemonic=Mnemonics()\\nfor i in range(26):\\n    mnemonic.register(chr(ord(\'\'a\'\')+i),i)\\nfor i in range(26):\\n    mnemonic.register(chr(ord(\'\'A\'\')+i),i)\\nfor i in range(10):\\n    mnemonic.register(\'\'%d\'\'%%i,26+i)\\nfor i in \'\'!\'\'#$%\'\'&\'\'()*+,-./:;<=>?@[\\\\]^^_\'\'{|}~\'\'\\n    mnemonic.register(i,punctuation.index(i)+36)\\nPourings=functools.partial(pourings,base=62)\\n\\nstdin.readline\\ndef I():\\n    return int(stdin.readline())\\ndef LIST():\\n    return [int(x) for x in stdin.readline().split()]\\n\\ndef S():\\n    return stdin.readline()[:-1]\\n\\nfrom random import randint,seed\\nseed(0)\\nt=I()\\nfor _ in range(t):\\n    # 25 10^3\\n    nm=LIST()\\n    n,k=nm\\n    a=LIST()\\n    total=0\\n    f_max=lambda x:Pourings(x,randint(1,k-1),a)[-1]\\n    f_min=lambda x:Pourings(x,randint(1,k-1),a)[0]\\n    l=1<<60\\n    r=0\\n    for ai in a:\\n        l=min(l,ai)\\n        r=max(r,ai)\\n    for _ in range(40):\\n        m=(l+r)//2\\n        if f_max(m)>=f_min(m):\\n            r=m\\n        else:\\n            l=m+1\\n    print(r)\\n    \\n# def pout(*args,**kwargs):\\n#     for i in args:\\n#         stderr.write(str(i))\\n#         stderr.write(\'\' \'\')\\n#     stderr.write(\'\\\\n\')\\n# # 1<=k<=n-1\\n# k=n-1->total pour=total init,make all equal to 0\\n# pour+1=total init-1->1 mean>0,all can be positive\\n\\n# ceil((1+2+..+k)/k)=k/2+1->all pour to the barrel which has the most water\\n\\n# 2*10^5 is total of n in all test cases"]',
  'pr': 3},
 {'correct': '["for _ in range(int(input())):\\n    s = input()\\n    p = [i for i in s.split(\\"0\\") if i!=\\"\\"]\\n    p.sort(reverse=True)\\n    ans = 0\\n    for i in range(0,len(p),2):\\n        ans+=len(p[i])\\n    print(ans)\\n\\n", "for _ in range(int(input())):\\n    s=[len(i)for i in input().split(\'0\')]\\n    s.sort()\\n    print(sum(s[-1::-2]))", "for _ in range(int(input())):\\n    s = input()\\n    t = [i for i in s.split(\\"0\\") if i!=\\"\\"]\\n    t.sort(reverse=True)\\n    cnt=0\\n    for i in range(0,len(t),2):\\n        cnt+=len(t[i])\\n    print(cnt)", "for _ in range(int(input())):\\n    s = input()\\n    ar = []\\n    cur = 0\\n    for c in s:\\n        if c == \\"1\\":\\n            cur += 1\\n        else:\\n            ar.append(cur)\\n            cur = 0\\n    if cur != 0:\\n        ar.append(cur)\\n    ar.sort()\\n    ar.reverse()\\n    print(sum(ar[::2]))\\n", "for nt in range(int(input())):\\n\\ts = input()\\n\\tn = len(s)\\n\\tif s[0]==\\"1\\":\\n\\t\\tcount = 1\\n\\telse:\\n\\t\\tcount = 0\\n\\tgroups = []\\n\\tfor i in range(1,n):\\n\\t\\tif s[i]==\\"1\\":\\n\\t\\t\\tcount += 1\\n\\t\\telse:\\n\\t\\t\\tif count:\\n\\t\\t\\t\\tgroups.append(count)\\n\\t\\t\\tcount = 0\\n\\tif count:\\n\\t\\tgroups.append(count)\\n\\tgroups.sort(reverse=True)\\n\\tans = 0\\n\\tfor i in range(0,len(groups),2):\\n\\t\\tans += groups[i]\\n\\tprint (ans)\\n", "def solv():\\n\\ts=list(map(int,input()))\\n\\tv=[]\\n\\tsm=0\\n\\tfor n in s:\\n\\t\\tif n:\\n\\t\\t\\tsm+=1\\n\\t\\telse:\\n\\t\\t\\tv.append(sm)\\n\\t\\t\\tsm=0\\n\\tif sm:v.append(sm)\\n\\tv.sort(reverse=True)\\n\\n\\tres=0\\n\\n\\tfor n in range(0,len(v),2):res+=v[n]\\n\\tprint(res)\\n\\nfor _ in range(int(input())):solv()", "import math\\nt=int(input())\\nfor w in range(t):\\n    s=sorted(input().split(\'0\'),reverse=True)\\n    c=0\\n    for i in range(0,len(s),2):\\n        c+=len(s[i])\\n    print(c)", "from itertools import groupby\\n\\nt = int(input())\\n\\nfor _ in range(t):\\n    s = input()\\n    l = []\\n    for k, v in groupby(s):\\n        if k == \'1\':\\n            l.append(len(list(v)))\\n    l.sort(reverse=True)\\n    n = len(l)\\n    res = 0\\n    for i in range(0, n, 2):\\n        res += l[i]\\n    print(res)\\n", "for _ in range(int(input())):\\n    s = input()\\n    x = sorted(len(i) for i in s.split(\'0\') if len(i) > 0)\\n\\n    print(max(sum(x[::2]), sum(x[1::2])))", "from sys import stdin,stdout\\nfrom math import sqrt,gcd,ceil,floor,log2,log10,factorial,cos,acos,tan,atan,atan2,sin,asin,radians,degrees,hypot\\nfrom bisect import insort, insort_left, insort_right, bisect_left, bisect_right, bisect\\nfrom array import array\\nfrom functools import reduce\\nfrom itertools import combinations, combinations_with_replacement, permutations\\nfrom fractions import Fraction\\nfrom random import choice,getrandbits,randint,random,randrange,shuffle\\nfrom re import compile,findall,escape\\nfrom statistics import mean,median,mode\\nfrom heapq import heapify,heappop,heappush,heappushpop,heapreplace,merge,nlargest,nsmallest\\n\\nfor test in range(int(stdin.readline())):\\n    s=input()\\n    l=findall(r\'1+\',s)\\n    lengths=[len(i) for i in l]\\n    lengths.sort(reverse=True)\\n    alice=0\\n    for i in range(0,len(lengths),2):\\n        alice+=lengths[i]\\n    print(alice)", "import sys\\ninput = sys.stdin.readline\\nT = int(input())\\n\\nfor t in range(T):\\n    s = input()[:-1]\\n\\n    counts = []\\n    current = 0\\n    for c in s:\\n        if c == \'1\':\\n            current += 1\\n        else:\\n            counts.append(current)\\n            current = 0\\n    if current:\\n        counts.append(current)\\n\\n    res = 0\\n    counts = sorted(counts, reverse=True)\\n    for i in range(len(counts)):\\n        if 2*i >= len(counts):\\n            break\\n        res += counts[2*i]\\n    print(res)\\n", "import sys\\nimport math\\ndef II():\\n\\treturn int(sys.stdin.readline())\\n\\ndef LI():\\n\\treturn list(map(int, sys.stdin.readline().split()))\\n\\ndef MI():\\n\\treturn map(int, sys.stdin.readline().split())\\n\\ndef SI():\\n\\treturn sys.stdin.readline().strip()\\nt = II()\\nfor q in range(t):\\n\\ts = SI()\\n\\ta = []\\n\\tcount = 0\\n\\tfor i in range(len(s)):\\n\\t\\tif s[i] == \\"1\\":\\n\\t\\t\\tcount+=1\\n\\t\\telse:\\n\\t\\t\\ta.append(count)\\n\\t\\t\\tcount = 0\\n\\ta.append(count)\\n\\ta.sort(reverse=True)\\n\\tprint(sum(a[0:len(a):2]))", "from math import *\\nfrom collections import *\\nfrom random import *\\nfrom decimal import Decimal\\nfrom heapq import *\\nfrom bisect import *\\nimport sys\\ninput=sys.stdin.readline\\nsys.setrecursionlimit(10**5)\\ndef lis():\\n    return list(map(int,input().split()))\\ndef ma():\\n    return list(map(int,input().split()))\\ndef inp():\\n    return int(input())\\ndef st1():\\n    return input().rstrip(\'\\\\n\')\\nt=inp()\\nwhile(t):\\n    t-=1\\n    #n=inp()\\n    a=st1()\\n    oe=[]\\n    c=0\\n    for i in a:\\n        if(i==\'1\'):\\n            c+=1\\n        else:\\n            if(c!=0):\\n                oe.append(c)\\n                c=0\\n    if(c):\\n        oe.append(c)\\n    s=0\\n    oe.sort(reverse=True)\\n    for i in range(len(oe)):\\n        if(i%2==0):\\n            s+=oe[i]\\n    print(s)\\n        \\n", "for _ in range(int(input())):\\n    s = input() + \'0\'\\n    A = []\\n    tr = False\\n    x = 0\\n    for i in range(len(s)):\\n        if s[i] == \'1\':\\n            if tr:\\n                x += 1\\n            else:\\n                tr = True\\n                x = 1\\n        else:\\n            if tr:\\n                tr = False\\n                A.append(x)\\n    A.sort(reverse=True)\\n    Ans = 0\\n    for i in range(len(A)):\\n        if i % 2 == 0:\\n            Ans += A[i]\\n    print(Ans)", "t = int(input())\\nwhile t:\\n    s = input()\\n    arr = []\\n    k = 0\\n    for i in s:\\n        if i == \'1\':\\n            k += 1\\n        else:\\n            arr.append(k)\\n            k = 0\\n    if k:\\n        arr.append(k)\\n    arr.sort(reverse=True)\\n    ans = 0\\n    for i in range(0, len(arr), 2):\\n        ans += arr[i]\\n    print(ans)\\n    t -= 1\\n", "import sys\\ninput = sys.stdin.readline\\n\\nt = int(input())\\nfor _ in range(t):\\n    x = input().rstrip()\\n    \\n    arr = []\\n    \\n    c = 0\\n    for char in x:\\n        if char==\'1\':\\n            c+=1\\n        else:\\n            arr.append(c)\\n            c = 0\\n            \\n    arr.append(c)\\n    arr.sort()\\n    arr.reverse()\\n    \\n    ans = 0\\n    for i in range(0,len(arr),2):\\n        ans += arr[i]\\n        \\n    print(ans)", "import sys\\ninput = sys.stdin.readline\\n\\nt=int(input())\\nfor tests in range(t):\\n    S=input().strip()+\\"0\\"\\n\\n    L=[]\\n\\n    NOW=0\\n    for s in S:\\n        if s==\\"0\\":\\n            L.append(NOW)\\n            NOW=0\\n        else:\\n            NOW+=1\\n\\n    L.sort(reverse=True)\\n\\n    ANS=0\\n\\n    for i in range(0,len(L),2):\\n        ANS+=L[i]\\n\\n    print(ANS)\\n        \\n", "for _ in range (int(input())):\\n    s=input()\\n    a = []\\n    flag = 0\\n    count = 0\\n    for i in range (len(s)):\\n        if s[i]==\'1\':\\n            count+=1\\n        else:\\n            a.append(count)\\n            count=0\\n        if i==len(s)-1 and count!=0:\\n            a.append(count)\\n    a.sort(reverse=True)\\n    ans = 0\\n    for i in range(len(a)):\\n        if i%2==0:\\n            ans+=a[i]\\n    print(ans)", "for t in range(int(input())):\\n\\ts = input()\\n\\tlast = -1\\n\\tnum = []\\n\\tn = len(s)\\n\\tfor i in range(n):\\n\\t\\tif (s[i] == \\"0\\"):\\n\\t\\t\\tif (i - last - 1 > 0):\\n\\t\\t\\t\\tnum.append(i - last - 1)\\n\\t\\t\\tlast = i\\n\\tif (n - last - 1 > 0):\\n\\t\\tnum.append(n - last - 1)\\n\\tnum = sorted(num)[::-1]\\n\\tans = 0\\n\\tfor i in range(0, len(num), 2):\\n\\t\\tans += num[i]\\n\\tprint(ans)", "for test in range(int(input())):\\n    s = input()\\n    a = []\\n    now = 0\\n    n = len(s)\\n    for i in range(n):\\n        if s[i] == \\"0\\":\\n            if now > 0:\\n                a.append(now)\\n            now = 0\\n        else:\\n            now += 1\\n    if now > 0:\\n        a.append(now)\\n    a.sort(reverse=True)\\n    ans = 0\\n    for i in range(0, len(a), 2):\\n        ans += a[i]\\n    print(ans)", "for _ in range(int(input())):\\n    s = input()\\n\\n    ones = []\\n    cnt = 0\\n    for i in s:\\n        if i == \'1\':\\n            cnt += 1\\n        else:\\n            if cnt != 0:\\n                ones.append(cnt)\\n                cnt = 0\\n    if cnt != 0:\\n        ones.append(cnt)\\n\\n    ones.sort(reverse=True)\\n    print(sum(ones[::2]))\\n", "from collections import defaultdict as dd\\nimport math\\nimport sys\\ninput=sys.stdin.readline\\ndef nn():\\n\\treturn int(input())\\n\\ndef li():\\n\\treturn list(input())\\n\\ndef mi():\\n\\treturn list(map(int, input().split()))\\n\\ndef lm():\\n\\treturn list(map(int, input().split()))\\n\\ndef solve():\\n\\ts = input()\\n\\n\\tsets = []\\n\\tstreak = 0\\n\\tfor i in range(len(s)):\\n\\t\\tif s[i]==\'1\':\\n\\t\\t\\tstreak+=1\\n\\t\\telse:\\n\\t\\t\\tif streak>0:\\n\\t\\t\\t\\tsets.append(streak)\\n\\t\\t\\t\\tstreak=0\\n\\tif streak>0:\\n\\t\\tsets.append(streak)\\n\\t\\tstreak=0\\n\\n\\tsets.sort(reverse=True)\\n\\n\\tprint(sum(sets[::2]))\\n\\n\\nq=nn()\\nfor _ in range(q):\\n\\tsolve()\\n", "t = int(input())\\n\\nfor _ in range(t):\\n    s = [int(i) for i in input().strip()]\\n    n = len(s)\\n    bckt = []\\n    ct = 0\\n    \\n    for i in range(n):\\n        if s[i]:\\n            ct += 1\\n        else:\\n            if ct:\\n                bckt.append(ct)\\n                ct = 0\\n    \\n    if ct:\\n        bckt.append(ct)\\n        \\n    bckt.sort(reverse=True)\\n    print(sum(bckt[::2]))", "for i in range(int(input())):\\n\\tip=list(map(int,input()))\\n\\tones=[]\\n\\ttot=0\\n\\tfor i in ip:\\n\\t\\tif i==1:\\n\\t\\t\\ttot+=1\\n\\t\\telse:\\n\\t\\t\\tones.append(tot)\\n\\t\\t\\ttot=0\\n\\tif tot:ones.append(tot)\\n\\tones.sort(reverse=True)\\n\\tans=0\\n\\tfor i in range(0,len(ones),2):\\n\\t\\tans+=ones[i]\\n\\tprint(ans)", "#BINOD\\nimport math\\ntest = int(input())\\nfor t in range(test):\\n    s = input()\\n    n = len(s)\\n    A = []\\n    o=0\\n    for i in range(n):\\n        if(s[i]==\'1\'):\\n            o+=1\\n        else:\\n            A.append(o)\\n            o=0\\n    if(s[n-1]==\'1\'):\\n        A.append(o)\\n    A.sort(reverse = True)\\n    ans = 0\\n    for i in range(0,len(A),2):\\n        ans += A[i]\\n    print(ans)\\n\\n\\n\\n\\n#Binod\\n", "for _ in range(int(input())):\\n    data = list(map(int,list(input())))\\n    fl = False\\n    data.append(\\"&\\")\\n    l = 0\\n    st = []\\n    for i in range(len(data)):\\n        if fl and data[i] == 1:\\n            l+=1\\n            continue\\n        if fl and data[i]!=1:\\n            st.append(l)\\n            l = 0\\n            fl = False\\n            continue\\n        if not fl and data[i] == 1:\\n            l = 1\\n            fl = True\\n    st.sort(reverse=True)\\n    c1 = 0\\n    for i in range(0,len(st),2):\\n        c1+=st[i]\\n    print(c1)", "import math\\nfrom collections import deque\\nfrom sys import stdin, stdout\\nfrom string import ascii_letters\\ninput = stdin.readline\\n#print = stdout.write\\nletters = ascii_letters[:26]\\n \\nfor _ in range(int(input())):\\n    arr = list(map(int, input().strip()))\\n    lens = []\\n    count = 0\\n    for i in arr:\\n        if i == 0:\\n            if count > 0:\\n                lens.append(count)\\n            count = 0\\n        else:\\n            count += 1\\n    if count > 0:\\n        lens.append(count)\\n    lens.sort(reverse=True)\\n    res = 0\\n    for i in range(0, len(lens), 2):\\n        res += lens[i]\\n    print(res)\\n"]',
  'predicted': '["def flag_s():\\n    return True\\n    \\n    import sys\\nsys.setrecursionlimit(10**7)\\nflag_s()\\n\\nfrom bisect import bisect_right, bisect_left, insort\\nfrom collections import deque, defaultdict\\nfrom heapq import heapify, heappop, heappush,heapreplace\\nfrom itertools import product, combinations\\nfrom math import gcd, sqrt, ceil, floor, factorial\\nfrom functools import lru_cache,reduce\\n\\nmod=10**9+7\\ninf=float(\'inf\')\\n\\n\\n\\n\\nT=int(input())\\nfor _ in range(T):\\n    S=input()\\n    \\n    count=0\\n    ans=0\\n    \\n    pre=S[0]\\n    for s in S:\\n        if s==pre:\\n            count+=1\\n        else:\\n            if pre==\'1\':\\n                ans+=count\\n            pre=s\\n            count=1\\n    \\n    if pre==\'1\':\\n        ans+=count\\n\\n    print(ans)\\n    \\n\\n    \\n", "import sys\\ninput = sys.stdin.readline\\n\\nt = int(input())\\nfor _ in range(t):\\n    s = input().rstrip()\\n    ans = s.count(\'1\')\\n    for i in range(len(s)):\\n        if s[i] == \'0\' and i > 0 and i < len(s)-1 and s[i-1] == s[i+1] == \'1\':\\n            ans += 1\\n    print (ans)\\n", "def __builtin():\\n    import sys\\n    class __S(object):\\n        def __init__(self):\\n            self.io = sys.stdin\\n            self.nm = sys.maxint\\n            self.buf = self.line = None\\n            self.ln = 0\\n        def __iter__(self):\\n            return self\\n        def __next__(self):\\n            return self.readline()\\n        def readline(self):\\n            if self.line is None:\\n                self.line = self.io.readline()\\n            if self.line == \'\':\\n                raise StopIteration\\n            buf = self.line\\n            self.line = None\\n            return buf\\n        def bufferline(self):\\n            if self.buf is None:\\n                self.buf = self.io.read()\\n            lines = self.buf.splitlines()\\n            if self.ln >= len(lines):\\n                raise StopIteration\\n            line = lines[self.ln]\\n            self.ln += 1\\n            return line\\n    sys.modules[\'builtins\'].__S = __S\\n    sys.modules[\'builtins\'].input = __S().bufferline\\n    sys.modules[\'builtins\'].next = __S().bufferline\\n__builtin()\\n\\ndef LI():\\n    return list(map(int, input().split()))\\n\\ndef II():\\n    return int(input())\\n\\ndef MI(rows=True):\\n    lines = input()\\n    if not lines:\\n        return {}\\n    mat = [list(map(int, line.split())) for line in lines.splitlines()]\\n    return mat if rows else list(zip(*mat))\\n\\ndef SI():\\n    return input()[:-1]\\n\\nfor _ in range(II()):\\n    s = SI()\\n    alice = bob = 0\\n    prev = None\\n    cnt = 0\\n    for c in s:\\n        if c == \'1\':\\n            if prev == \'1\':\\n                cnt += 1\\n            else:\\n                cnt = 1\\n        else:\\n            if prev == \'1\':\\n                if cnt % 2:\\n                    alice += cnt // 2 + 1\\n                    bob += cnt // 2\\n                else:\\n                    alice += cnt // 2\\n                    bob += cnt // 2\\n                cnt = 0\\n        prev = c\\n    if prev == \'1\':\\n        if cnt % 2:\\n            alice += cnt // 2 + 1\\n            bob += cnt // 2\\n        else:\\n            alice += cnt // 2\\n            bob += cnt // 2\\n    print(alice)\\n", "from collections import deque\\nimport math\\narr1 = []\\nt = int(input())\\nfor r in range(t):\\n    s = input()\\n    arr1.append(s)\\nfor r in arr1:\\n    a = 0\\n    b = 0\\n    c = 0\\n    d = 0\\n    for i in range(len(r)):\\n        if r[i] == \\"1\\":\\n            if c == 0:\\n                a += 1\\n            c += 1\\n        else:\\n            d = max(d,c)\\n            c = 0\\n    d = max(d,c)\\n    b = d\\n    if d % 2 == 0:\\n        a += d//2\\n        b += d//2\\n    else:\\n        a += d//2 + 1\\n        b += d//2\\n    print(a)\\n", "from collections import deque\\nimport math\\narr1 = []\\nt = int(input())\\nfor r in range(t):\\n    s = input()\\n    arr1.append(s)\\nfor r in arr1:\\n    a = 0\\n    c = 0\\n    for i in range(len(r)):\\n        if r[i] == \\"1\\":\\n            c += 1\\n        else:\\n            a = max(a,c)\\n            c = 0\\n    a = max(a,c)\\n\\n    if a % 2 == 0:\\n        print(int(a/2))\\n    else:\\n        print(int(a/2)+1))\\n", "for _ in range(int(input())):\\n    s = input()\\n    one = []\\n    l = 0\\n    for i in range(len(s)):\\n        if s[i] == \'1\':\\n            l += 1\\n        else:\\n            if l != 0:\\n                one.append(l)\\n            l = 0\\n    if l != 0:\\n        one.append(l)\\n    ans = 0\\n    for i in range(len(one)):\\n        if i % 2 == 0:\\n            ans += (one[i] + 1) // 2\\n        else:\\n            ans += one[i] // 2\\n    print(ans)", "t = int(input())\\nfor i in range(t):\\n    s = input()\\n    res = 0\\n    ones = 0\\n    zeros = 0\\n    while(zeros < len(s) and s[zeros] == \\"0\\"):\\n        zeros += 1\\n    while(ones + zeros < len(s) and s[ones + zeros] == \\"1\\"):\\n        ones += 1\\n    turn = 1\\n    while(zeros < len(s) or ones < len(s):\\n        if turn == 1:\\n            toTake = (ones - zeros + 1) // 2\\n            res += toTake\\n        else:\\n            toTake = ones // 2\\n        \\n        if toTake < 1 or zeros == len(s):\\n            break\\n        if turn == 1:\\n            ones -= toTake\\n        else:\\n            ones -= toTake\\n        turn = 3 - turn\\n    print(res)", "t = int(input())\\nfor i in range(t):\\n    s = input()\\n    k = 0\\n    r = []\\n    for j in range(len(s)):\\n        if s[j] == \'1\':\\n            k += 1\\n        elif k > 0:\\n            r.append(k)\\n            k = 0\\n    if k > 0:\\n        r.append(k)\\n    for j in range(len(r)):\\n        if j % 2 == 1:\\n            r[j] = int(r[j] / 2)\\n        else:\\n            r[j] = int((r[j] + 1) / 2)\\n    print(sum(r))\\n", "import math\\nimport sys\\nfrom collections import deque\\nfrom collections import defaultdict\\ninput = sys.stdin.readline\\ndef rui(N, A, func, ele):\\n  \\"\\"\\"ele: identity element\\"\\"\\"\\n  rui = [ele] * (N + 1)\\n  for i in range(1, N + 1):\\n    rui[i] = func(rui[i - 1], A[i - 1])\\n  return rui\\n\\ndef precin(A):\\n  stdin = sys.stdin.popen(\\"%s\\" % input, \\"r\\")\\n  sys.stdin = stdin\\n  inAIQ = A\\n  return inAIQ\\n#sys.setrecursionlimit(10**6)\\n#inf = float(\\"inf\\")\\nmod = 10**9 + 7\\n#YES = 1\\n#NO = 0\\nQ = 10**5 + 10\\n#r = precin(\\"1 29\\\\n1000101110001010010000000\\\\n\\", Q)\\n#for _ in range(2):\\nT = int(input())\\nfor test_case in range(T):\\n  S = list(input())[:-1]\\n  cnt = 0\\n  ans = 0\\n  prev = \\"0\\"\\n  for s in S:\\n    if s == prev:\\n      cnt += 1\\n    else:\\n      if prev == \\"1\\":\\n        if cnt % 2 == 0:\\n          ans += cnt // 2\\n        else:\\n          ans += cnt // 2 + 1\\n      prev = s\\n      cnt = 1\\n  if prev == \\"1\\":\\n    if cnt % 2 == 0:\\n      ans += cnt // 2\\n    else:\\n      ans += cnt // 2 + 1\\n  print(ans)\\n# 1 2  = 3\\n#      4", "t = int(input())\\nfor i in range(t):\\n    s = input()\\n    r = []\\n    k = 0\\n    for j in range(len(s)):\\n        if s[j] == \'1\':\\n            k += 1\\n        else:\\n            if k != 0:\\n                r.append(k)\\n            k = 0 \\n    if k != 0:\\n        r.append(k)\\n\\n    ans = 0\\n\\n    for j in range(len(r)):\\n        if j % 2 == 0:\\n            ans += (r[j] + 1) // 2 \\n        else:\\n            ans += r[j] // 2\\n    print(ans)\\n", "import math\\nimport sys\\nfrom collections import deque\\nfrom collections import defaultdict\\ninput = sys.stdin.readline\\ndef rui(N, A, func, ele):\\n  \\"\\"\\"ele: identity element\\"\\"\\"\\n  rui = [ele] * (N + 1)\\n  for i in range(1, N + 1):\\n    rui[i] = func(rui[i - 1], A[i - 1])\\n  return rui\\n\\ndef precin(A):\\n  stdin = sys.stdin.popen(\\"%s\\" % input, \\"r\\")\\n  sys.stdin = stdin\\n  inAIQ = A\\n  return inAIQ\\n#sys.setrecursionlimit(10**6)\\n#inf = float(\\"inf\\")\\nmod = 10**9 + 7\\n#YES = 1\\n#NO = 0\\nQ = 10**5 + 10\\n#r = precin(\\"1 29\\\\n1000101110001010010000000\\\\n\\", Q)\\n#for _ in range(2):\\nT = int(input())\\nfor test_case in range(T):\\n  S = list(input())[:-1]\\n  cnt = 0\\n  ans = 0\\n  prev = \\"0\\"\\n  for s in S:\\n    if s == prev:\\n      cnt += 1\\n    else:\\n      if prev == \\"1\\":\\n        ans += cnt // 2\\n      prev = s\\n      cnt = 1\\n  if prev == \\"1\\":\\n    ans += cnt // 2\\n  print(ans)\\n# 1 2  = 3\\n#      4", "N=int(input())\\nfor _ in range(N):\\n    S=input()\\n    one=0\\n    zero=0\\n    if S[0]==\'1\':\\n        one+=1\\n    else:\\n        zero+=1\\n    for i in range(1,len(S)):\\n        if S[i]!=S[i-1]:\\n            if S[i]==\'1\':\\n                one+=1\\n            else:\\n                zero+=1\\n    if zero>one:\\n        one,zero=zero,one\\n    if (one-zero)%2==0:\\n        print(zero+one//2)\\n    else:\\n        print(zero+one//2+1)\\n", "\\"\\"\\"\\nimport numpy as np\\nimport pandas as pd\\n\\"\\"\\"\\nimport sys\\nfrom collections import deque\\nfrom collections import defaultdict\\ninput = sys.stdin.readline\\ndef rui(N, A, func, ele):\\n  \\"\\"\\"ele: identity element\\"\\"\\"\\n  rui = [ele] * (N + 1)\\n  for i in range(1, N + 1):\\n    rui[i] = func(rui[i - 1], A[i - 1])\\n  return rui\\n\\ndef precin(A):\\n  stdin = sys.stdin.popen(\\"%s\\" % input, \\"r\\")\\n  sys.stdin = stdin\\n  inAIQ = A\\n  return inAIQ\\n#sys.setrecursionlimit(10**6)\\n#inf = float(\\"inf\\")\\nmod = 10**9 + 7\\n#YES = 1\\n#NO = 0\\nQ = 10**5 + 10\\n#r = precin(\\"1 29\\\\n1000101110001010010000000\\\\n\\", Q)\\n#for _ in range(2):\\nT = int(input())\\nfor test_case in range(T):\\n  S = input()[:-1]\\n  cnt = 0\\n  ans = 0\\n  prev = \\"0\\"\\n  for s in S:\\n    if s == prev:\\n      cnt += 1\\n    else:\\n      if prev == \\"1\\":\\n        ans += cnt // 2 + 1\\n      prev = s\\n      cnt = 1\\n  if prev == \\"1\\":\\n    ans += cnt // 2 + 1\\n  print(ans)\\n# 1 2  = 3\\n#      4\\n\\n#S = \\"100000111000000000000000000000011111100000111111111111000\\"[:-1]\\n#cnt = 0\\n#ans = 0\\n#prev = \\"0\\"\\n#for s in S:\\n#  if s == prev:\\n#    cnt += 1\\n#  else:\\n#    if prev == \\"1\\":\\n#      ans += cnt // 2 + 1\\n#    prev = s\\n#    cnt = 1\\n#if prev == \\"1\\":\\n#  ans += cnt // 2 + 1\\n#print(ans)\\n", "import sys\\ninput = lambda: sys.stdin.readline().strip()\\ndef ii():\\n    return int(input())\\ndef li():\\n    return list(map(int, input().split()))\\ndef mi(rows=True):\\n    lines = input()\\n    if not lines:\\n        return []\\n    mat = [list(map(int, line.split())) for line in lines.splitlines()]\\n    return mat if rows else list(zip(*mat))\\ndef mi1D(stmt=\\"such as x = le()\\"):\\n    return list(map(int, input().split()))\\ndef st1D(stmt=\\"such as x = le()\\"):\\n    return list(input())\\n\\nt = ii()\\nfor _ in range(t):\\n    s = st1D()\\n    ans = 0\\n    one = 0\\n    zero = 0\\n    while zero < len(s) and s[zero] == \\"0\\":\\n        zero += 1\\n    while one + zero < len(s) and s[one + zero] == \\"1\\":\\n        one += 1\\n    if one + zero == len(s):\\n        print( (one + 1) // 2 )\\n        continue\\n    turn = 1\\n    while one + zero < len(s):\\n        if turn == 1:\\n            toTake = (one - zero + 1) // 2\\n            ans += toTake\\n        else:\\n            toTake = one // 2\\n        \\n        if toTake < 1 or one + zero == len(s):\\n            break\\n        if turn == 1:\\n            one -= toTake\\n        else:\\n            one -= toTake\\n        turn = 1 - turn + 1\\n    print(ans)", "N=int(input())\\nfor _ in range(N):\\n    S=input()\\n    c=0\\n    s=0\\n    for i in range(len(S)):\\n        if S[i]==\'1\':\\n            if s%2==0:\\n                c+=1\\n            s+=1\\n    print(c)\\n", "import sys\\n\\ninput = sys.stdin.readline\\n\\n\\ndef main():\\n    t = int(input())\\n    for test in range(t):\\n        s = input().strip()\\n        ones = []\\n        cnt = 0\\n        for c in s:\\n            if c == \'1\':\\n                cnt += 1\\n            else:\\n                if cnt:\\n                    ones.append(cnt)\\',
  'pr': 9},
 {'correct': '["import sys\\nimport random\\nfrom fractions import Fraction\\nfrom math import *\\n \\ndef input():\\n    return sys.stdin.readline().strip()\\n \\ndef iinput():\\n    return int(input())\\n\\ndef finput():\\n    return float(input())\\n\\ndef tinput():\\n    return input().split()\\n\\ndef linput():\\n    return list(input())\\n \\ndef rinput():\\n    return list(map(int, tinput()))\\n\\ndef fiinput():\\n    return list(map(float, tinput()))\\n \\ndef rlinput():\\n    return list(map(int, input().split()))\\ndef trinput():\\n    return tuple(rinput())\\n\\ndef srlinput():\\n    return sorted(list(map(int, input().split())))\\n\\ndef NOYES(fl):\\n    if fl:\\n        print(\\"NO\\")\\n    else:\\n        print(\\"YES\\")\\ndef YESNO(fl):\\n    if fl:\\n        print(\\"YES\\")\\n    else:\\n        print(\\"NO\\")\\n    \\ndef main():\\n    n = iinput()\\n    #k = iinput() \\n    #m = iinput() \\n    #n = int(sys.stdin.readline().strip()) \\n    #n, k = rinput()\\n    #n, m = rinput()\\n    #m, k = rinput()\\n    #n, k, m = rinput()\\n    #n, m, k = rinput()\\n    #k, n, m = rinput()\\n    #k, m, n = rinput() \\n    #m, k, n = rinput()\\n    #m, n, k = rinput()\\n    q = [rlinput(), rlinput(), rlinput()]\\n    #q = linput()\\n    ans = q[0].copy()\\n    for i in range(1, n):\\n        if ans[i] == ans[i - 1]:\\n            ans[i] = q[1][i]\\n        if i == n - 1:\\n            o = 0\\n            while q[o][i] == ans[n - 2] or q[o][i] == ans[0]:\\n                o += 1\\n            ans[i] = q[o][i]\\n    print(*ans)\\n\\n        \\n\\n            \\n        \\n    \\n                \\n    \\n    \\n    \\n            \\n    \\n        \\n    \\n\\nfor i in range(iinput()):\\n    main()\\n", "for _ in range(int(input())):\\n  n=int(input())\\n  a=list(map(int,input().split()))\\n  b=list(map(int,input().split()))\\n  c=list(map(int,input().split()))\\n  p=a\\n  for i in range(n):\\n    if p[i]==p[(i+1)%n]:\\n      if p[i]!=b[i] and p[(i-1)%n]!=b[i]:p[i]=b[i]\\n      else:p[i]=c[i]\\n  print(*p)", "for __ in range(int(input())):\\n    n = int(input())\\n    ar1 = list(map(int, input().split()))\\n    ar2 = list(map(int, input().split()))\\n    ar3 = list(map(int, input().split()))\\n    ans = [ar1[0]]\\n    for i in range(1, n - 1):\\n        if ar1[i] != ans[-1]:\\n            ans.append(ar1[i])\\n        elif ar2[i] != ans[-1]:\\n            ans.append(ar2[i])\\n        elif ar3[i] != ans[-1]:\\n            ans.append(ar3[i])\\n    if ar1[-1] != ans[-1] and ar1[-1] != ans[0]:\\n        ans.append(ar1[-1])\\n    elif ar2[-1] != ans[-1] and ar2[-1] != ans[0]:\\n        ans.append(ar2[-1])\\n    elif ar3[-1] != ans[-1] and ar3[-1] != ans[0]:\\n        ans.append(ar3[-1])\\n    print(*ans)", "T = int(input())\\n\\nfor t in range(T):\\n    N = int(input())\\n    A = [int(_) for _ in input().split()]\\n    B = [int(_) for _ in input().split()]\\n    C = [int(_) for _ in input().split()]\\n\\n    R = []\\n\\n    for i in range(N):\\n        if i == 0:\\n            R.append(A[i])\\n            continue\\n        if i == N-1:\\n            if A[i] != R[0] and A[i] != R[-1]:\\n                R.append(A[i])\\n            elif B[i] != R[0] and B[i] != R[-1]:\\n                R.append(B[i])\\n            else:\\n                R.append(C[i])\\n            continue\\n\\n        if A[i] != R[-1]:\\n            R.append(A[i])\\n        else:\\n            R.append(B[i])\\n\\n    print(\' \'.join(map(str, R)))\\n", "gans = []\\nfor _ in range(int(input())):\\n    n = int(input())\\n    a = list(map(int, input().split()))\\n    b = list(map(int, input().split()))\\n    c = list(map(int, input().split()))\\n    ans = [a[0]]\\n    for i in range(1, n - 1):\\n        if a[i] != ans[i - 1]:\\n            ans.append(a[i])\\n        else:\\n            ans.append(b[i])\\n    if a[-1] != ans[-1] and a[-1] != ans[0]:\\n        ans.append(a[-1])\\n    elif b[-1] != ans[-1] and b[-1] != ans[0]:\\n        ans.append(b[-1])\\n    else:\\n        ans.append(c[-1])\\n    gans.append(\' \'.join(map(str, ans)))\\nprint(\'\\\\n\'.join(gans))\\n", "from math import *\\nfrom bisect import *\\nfrom collections import *\\nfrom random import *\\nfrom decimal import *\\nimport sys\\ninput=sys.stdin.readline\\ndef inp():\\n    return int(input())\\ndef st():\\n    return input().rstrip(\'\\\\n\')\\ndef lis():\\n    return list(map(int,input().split()))\\ndef ma():\\n    return list(map(int,input().split()))\\nt=inp()\\nwhile(t):\\n    t-=1\\n    n=inp()\\n    a=lis()\\n    b=lis()\\n    c=lis()\\n    r=[a[0]]\\n    for i in range(1,n):\\n        if(i==n-1):\\n            if(a[i]!=r[0] and a[i]!=r[-1]):\\n                r.append(a[i])\\n                continue\\n            if(b[i]!=r[0] and b[i]!=r[-1]):\\n                r.append(b[i])\\n                continue\\n            if(c[i]!=r[0] and c[i]!=r[-1]):\\n                r.append(c[i])\\n                continue\\n        if(a[i]!=r[-1]):\\n            r.append(a[i])\\n            continue\\n        if(b[i]!=r[-1]):\\n            r.append(b[i])\\n            continue\\n        if(c[i]!=r[-1]):\\n            r.append(c[i])\\n            continue\\n    print(*r)\\n        \\n        \\n        \\n"]',
  'predicted': '["t=int(input())\\n\\nfor _ in range(t):\\n    n=int(input())\\n    a=list(map(int, input().split()))\\n    b=list(map(int, input().split()))\\n    c=list(map(int, input().split()))\\n    d=[a[0],b[0],c[0]]\\n    for i in range(1,n):\\n        if i==n-1:\\n            if d[i-1]==a[i]:\\n                if d[0]!=a[i]:\\n                    d.append(a[i])\\n                else:\\n                    d.append(b[i])\\n            else:\\n                d.append(a[i])\\n\\n        else:\\n            if d[i-1]==a[i]:\\n                if d[i-2]!=b[i]:\\n                    d.append(b[i])\\n                else:\\n                    d.append(c[i])\\n            else:\\n                d.append(a[i])\\n    print(*d)", "\\n\\n\\n\\nfrom collections import deque\\ndef choose(a,b,c,prev):\\n    if a!=prev:\\n        return a\\n    elif b!=prev:\\n        return b\\n    else:\\n        return c\\n\\nt=int(input())\\nfor i in range(t):\\n    n=int(input())\\n    a=list(map(int, input().split()))\\n    b=list(map(int, input().split()))\\n    c=list(map(int, input().split()))\\n    p=[0]*n\\n    p[0]=a[0]\\n    p[1]=choose(b[1],c[1],a[1],p[0])\\n    for j in range(2,n):\\n        p[j]=choose(a[j],b[j],c[j],p[j-1])\\n    print(*p)\\n", "#                    _\\n#   _ __ ___   ___  | |_   ___   ___\\n#  | \'_ ` _ \\\\ / _ \\\\ | __| / _ \\\\ / _ \\\\\\n#  | | | | | |  __/ | |_ |  __/|  __/\\n#  |_| |_| |_|\\\\___|  \\\\__| \\\\___| \\\\___|\\n#\\n\\nfor _ in range(int(input())):\\n\\n    n = int(input())\\n\\n    a = list(map(int, input().split()))\\n    b = list(map(int, input().split()))\\n    c = list(map(int, input().split()))\\n\\n    ans = [0] * n\\n    \\n    ans[0] = a[0]\\n    \\n    if n > 1:\\n        if b[0] != a[1]:\\n            ans[1] = b[0]\\n        else:\\n            ans[1] = c[0]\\n\\n    for i in range(2, n):\\n\\n        if a[i-1] != ans[i-2]:\\n            ans[i] = a[i]\\n        elif b[i-1] != ans[i-2]:\\n            ans[i] = b[i]\\n        else:\\n            ans[i] = c[i]\\n\\n    print(\' \'.join(list(map(str, ans))))\\n", "t = int(input())\\nfor _ in range(t):\\n    n = int(input())\\n    A = tuple(int(i) for i in input().split())\\n    B = tuple(int(i) for i in input().split())\\n    C = tuple(int(i) for i in input().split())\\n    P = [0] * n\\n    P[0] = A[0]\\n    for i in range(1, n):\\n        if i + 1 == n:\\n            j = 0\\n        else:\\n            j = i + 1\\n        if P[i - 1] != A[i] and P[i - 1] != B[i] and P[i - 1] != C[i]:\\n            P[i] = A[i]\\n        elif P[i - 1] != B[i] and P[i - 1] != A[j] and P[i - 1] != C[i]:\\n            P[i] = B[i]\\n        else:\\n            P[i] = C[i]\\n    print(*P)", "for _ in range(int(input())):\\n    n = int(input())\\n    a = list(map(int, input().split()))\\n    b = list(map(int, input().split()))\\n    c = list(map(int, input().split()))\\n    d = [a[0]]\\n    for i in range(1, n):\\n        if i == n - 1:\\n            if d[i - 1] != a[i] and d[0] != a[i]:\\n                d.append(a[i])\\n            elif d[i - 1] != b[i] and d[0] != b[i]:\\n                d.append(b[i])\\n            else:\\n                d.append(c[i])\\n        else:\\n            if d[i - 1] != a[i]:\\n                d.append(a[i])\\n            elif d[i - 1] != b[i]:\\n                d.append(b[i])\\n            else:\\n                d.append(c[i])\\n    print(*d))\\n\\n", "t = int(input())\\nfor _ in range(t):\\n    n = int(input())\\n    a = list(map(int, input().split()))\\n    b = list(map(int, input().split()))\\n    c = list(map(int, input().split()))\\n    d = [0] * n\\n    if n == 3:\\n        d[0] = a[0]\\n        if d[0] != b[1] and d[0] != c[1]:\\n            d[1] = b[1]\\n        else:\\n            d[1] = c[1]\\n        d[2] = c[2] if d[0] != c[2] and d[1] != c[2] else (a[2] if d[1] != a[2] else b[2])\\n    else:\\n        d[0] = a[0]\\n        d[1] = b[1]\\n        for i in range(2, n):\\n            if d[i - 1] != a[i]:\\n                d[i] = a[i]\\n            elif d[i - 1] != b[i]:\\n                d[i] = b[i]\\n            else:\\n                d[i] = c[i]\\n    print(*d)\\n", "import numpy as np\\nt=int(input())\\nfor i in range(t):\\n    n=int(input())\\n    a=list(map(int,input().split()))\\n    b=list(map(int,input().split()))\\n    c=list(map(int,input().split()))\\n    d=np.zeros(n)\\n    d[0]=a[0]\\n    d[1]=b[1]\\n    for j in range(2,n):\\n        if d[j-1]!=a[j]:\\n            d[j]=a[j]\\n        elif d[j-1]!=b[j]:\\n            d[j]=b[j]\\n        else:\\n            d[j]=c[j]\\n    print(\' \'.join(map(str,d.astype(int))))\\n", "import sys\\ninput = sys.stdin.readline\\nt = int(input())\\nfor _ in range(t):\\n    n = int(input())\\n    a = list(map(int, input().split()))\\n    b = list(map(int, input().split()))\\n    c = list(map(int, input().split()))\\n    d = [0] * n\\n    for i in range(n):\\n        if i == 0:\\n            d[i] = a[i]\\n        elif i == n - 1:\\n            if d[i - 1] != a[i] and d[0] != a[i]:\\n                d[i] = a[i]\\n            elif d[i - 1] != b[i] and d[0] != b[i]:\\n                d[i] = b[i]\\n            else:\\n                d[i] = c[i]\\n        else:\\n            if d[i - 1] != a[i]:\\n                d[i] = a[i]\\n            elif d[i - 1] != b[i]:\\n                d[i] = b[i]\\n            else:\\n                d[i] = c[i]\\n    print(*d)", "from math import *\\nfrom collections import *\\nt=int(input())\\nfor _ in range(t):\\n    n=int(input())\\n    a=list(map(int,input().split()))\\n    b=list(map(int,input().split()))\\n    c=list(map(int,input().split()))\\n    ans=[0 for i in range(n)]\\n    ans[0]=a[0]\\n    if n>1:\\n        if b[0]!=ans[0]:\\n            ans[1]=b[0]\\n        else:\\n            ans[1]=c[0]\\n    for i in range(2,n):\\n        if ans[i-1]!=a[i]:\\n            ans[i]=a[i]\\n        elif ans[i-1]!=b[i]:\\n            ans[i]=b[i]\\n        else:\\n            ans[i]=c[i]\\n    print(*ans)\\n", "from itertools import islice\\n\\ndef read_ints():\\n    return map(int, input().split())\\n\\n\\nt = int(input())\\nfor _ in range(t):\\n    n = int(input())\\n    a = list(read_ints())\\n    b = list(read_ints())\\n    c = list(read_ints())\\n\\n    p = [a[0]]\\n    for i in range(1, n):\\n        if i == n-1:\\n            prev = p[0]\\n        else:\\n            prev = p[-1]\\n\\n        if prev != a[i] and p[-1] != b[i]:\\n            p.append(a[i])\\n        elif prev != a[i] and p[-1] != c[i]:\\n            p.append(a[i])\\n        else:\\n            p.append(b[i])\\n\\n    print(*p)", "from sys import stdin\\ndef lst():\\n    return [int(w) for w in stdin.readline().split()]\\n\\nt, =int(stdin.readline()),\\nfor k in range(t):\\n    n, =int(stdin.readline()),\\n    a = lst();b = lst();c = lst()\\n    p = [0] * n\\n    p[0] = a[0]\\n    p[1] = b[1]\\n    for i in range(2, n):\\n        if p[i-1] != a[i]:\\n            p[i] = a[i]\\n        elif p[i-1] != b[i]:\\n            p[i] = b[i]\\n        else:\\n            p[i] = c[i]\\n    if p[0] == p[-1]:\\n        if p[-2] != a[-1]:\\n            p[-1] = a[-1]\\n        elif p[-2] != b[-1]:\\n            p[-1] = b[-1]\\n        else:\\n            p[-1] = c[-1]\\n    print(*p)", "m = 100\\n\\nt = int(input())\\n\\nfor caso in range(t):\\n    n = int(input())\\n    A = list(map(int, input().split()))\\n    B = list(map(int, input().split()))\\n    C = list(map(int, input().split()))\\n\\n    sol = [0 for i in range(n)]\\n\\n    for i in range(n):\\n        if i == 0:\\n            sol[i] = A[i]\\n        elif i == n-1:\\n            if sol[i-1] != A[i] and sol[0] != A[i]:\\n                sol[i] = A[i]\\n            elif sol[i-1] != B[i] and sol[0] != B[i]:\\n                sol[i] = B[i]\\n            else:\\n                sol[i] = C[i]\\n        else:\\n            if sol[i-1] != A[i]:\\n                sol[i] = A[i]\\n            elif sol[i-1] != B[i]:\\n                sol[i] = B[i]\\n            else:\\n                sol[i] = C[i]\\n\\n    print(*sol)", "import sys\\nimport math\\nfrom collections import defaultdict\\nfrom collections import deque\\nfrom itertools import combinations\\nfrom itertools import permutations\\ninput = sys.stdin.readline\\n\\ndef getIntList():\\n    return list(map(int, input().split()))\\n\\ndef checkAdjacent(p):\\n    for i in range(len(p)):\\n        if p[i] == p[(i+1) % len(p)]:\\n            return False\\n    return True\\n\\nt = int(input())\\nfor _ in range(t):\\n    n = int(input())\\n    a = getIntList()\\n    b = getIntList()\\n    c = getIntList()\\n    p = [0] * n\\n    for i in range(n):\\n        if i == 0:\\n            p[i] = a[i]\\n        elif i == 1:\\n            if a[i] != p[i-1]:\\n                p[i] = a[i]\\n            else:\\n                p[i] = b[i]\\n        else:\\n            if a[i] != p[i-1] and a[i] != p[i-2]:\\n                p[i] = a[i]\\n            elif b[i] != p[i-1] and b[i] != p[i-2]:\\n                p[i] = b[i]\\n            else:\\n                p[i] = c[i]\\n    print (*p)", "import sys\\ninput = sys.stdin.readline\\n\\nfor _ in range(int(input())):\\n\\tn = int(input())\\n\\ta = list(map(int, input().split()))\\n\\tb = list(map(int, input().split()))\\n\\tc = list(map(int, input().split()))\\n\\tans = [0] * n\\n\\tfor i in range(n):\\n\\t\\tr = i - 1\\n\\t\\tif i == 0:\\n\\t\\t\\tr = n - 1\\n\\t\\telif i == 1:\\n\\t\\t\\tr = 0\\n\\t\\tans[i] = (set([a[i], b[i], c[i]]) - {ans[r]}).pop()\\n\\tprint(*ans)", "import sys\\ndef rl():\\n    return [int(w) for w in rs()]\\n\\ndef rt():\\n    return tuple(int(w) for w in rs())\\n\\ndef rs():\\n    return sys.stdin.readline().rstrip(\'\\n\')\\n\\nk, =  map(int, rs().split())\\nfor tn in range(k):\\n    lg = 0\\n    n, =  g =  rs()\\n    while 3**lg <= n: lg += 1\\n    A = [[0]*3 for i in range(n)]\\n    B = [[0]*3 for i in range(n)]\\n    C = [[0]*3 for i in range(n)]\\n    for i in range(n):\\n        A[i] = rt()\\n    for i in range(n):\\n        B[i] = rt()\\n    for i in range(n):\\n        C[i] = rt()\\n    P = [A[0]]\\n    for i in range(1,n):\\n        P.append(B[i])\\n    P[-1] = C[0]\\n    for i in range(1,n-1):\\n        for j in range(3):\\n            for k in range(3):\\n                if P[i][j] != P[i-1][k] and P[i][j] != P[i+1][k]:\\n                    P[i] = P[i][j]\\n                    break\\n    for i in range(3):\\n        if P[0][i] != P[-1][k] and P[0][i] != P[-2][k] :\\n            P[-1] = P[0][i]\\n            break\\n    print(\'\')\\n    for i in P:\\n        print(*i)", "for t in range(int(input())):\\n    n = int(input())\\n    a = list(map(int, input().split()))\\n    b = list(map(int, input().split()))\\n    c = list(map(int, input().split()))\\n    d = [0 for i in range(n)]\\n    for i in range(n):\\n        if i == 0:\\n            d[i] = a[i]\\n        elif i == n - 1:\\n            if d[i - 1] != a[i] and d[0] != a[i]:\\n                d[i] = a[i]\\n            elif d[i - 1] != b[i] and d[0] != b[i]:\\n                d[i] = b[i]\\n            else:\\n                d[i] = c[i]\\n        else:\\n            if d[i - 1] != a[i]:\\n                d[i] = a[i]\\n            elif d[i - 1] != b[i]:\\n                d[i] = b[i]\\n            else:\\n                d[i] = c[i]\\n    print(*d)\\n", "t = int(input())\\nfor f in range(t):\\n    n = int(input())\\n    a = [int(i) for i in input().split()]\\n    b = [int(i',
  'pr': 2}]

# Create a DataFrame from the list
#df = pd.DataFrame(data)
df = pd.DataFrame(dataset_test)

# Specify the output file name
output_file = "/Users/fabiomar/Documents/GitHub/llm-quality-research/core/content/data/output.csv"

# Save the DataFrame to a CSV file
df.to_csv(output_file, index=False)

print(f"CSV file '{output_file}' has been created with the following content:")
print(df)


CSV file '/Users/fabiomar/Documents/GitHub/llm-quality-research/core/content/data/output.csv' has been created with the following content:
                                             correct  \
0  ["import sys\ndef I():\n    return sys.stdin.r...   
1  ["def solve():\n    n, k = map(int,input().spl...   
2  ["for _ in range(int(input())):\n    s = input...   
3  ["import sys\nimport random\nfrom fractions im...   

                                           predicted  pr  
0  ["for _ in range(int(input())):\n    n = int(i...   7  
1  ["from sys import stdin,stderr\ndef msg(*args,...   3  
2  ["def flag_s():\n    return True\n    \n    im...   9  
3  ["t=int(input())\n\nfor _ in range(t):\n    n=...   2  
